# ♟️ AI Chess Commentator: Stage 4 QLoRA Fine-Tuning on Kaggle GPU

Fine-tune **microsoft/Phi-3.5-mini-instruct** (3.8B, MIT License) to generate natural-language chess commentary grounded in Stockfish evaluations and tactical taxonomy classifications.

### Recommended Kaggle Settings:
- **Accelerator**: GPU T4 x2 or GPU A100
- **Persistence**: Files only (or Variables and Files)
- **Internet**: On

In [ ]:
# Step 1: Install Dependencies
!apt-get update -qq && apt-get install -y -qq stockfish
!pip install -q bitsandbytes peft python-chess


In [ ]:
# Step 2: Unpack Embedded Dataset & Source Code
import base64, io, tarfile, os, sys

B64_DATA = "..."

print("Unpacking datasets and src package...")
raw = base64.b64decode(B64_DATA)
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(".")
print("Unpacked successfully!")
if "src" not in sys.path:
    sys.path.insert(0, "src")
print("Python path configured.")


In [ ]:
# Step 3: Verify GPU & Stockfish
import torch, shutil
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
stockfish_path = shutil.which("stockfish") or "/usr/games/stockfish"
print(f"Stockfish Path: {stockfish_path}")

In [ ]:
# Step 4: Configure Training Parameters
from chess_commentator.training.config import TrainingConfig

config = TrainingConfig(
    model_id="microsoft/Phi-3.5-mini-instruct",
    train_file="data/commentary_sft_train.jsonl",
    val_file="data/commentary_sft_val.jsonl",
    output_dir="/kaggle/working/phi35_chess_commentator_adapter",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    max_seq_length=512,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    optim="paged_adamw_8bit",
    seed=42,
)
print(f"Configured model: {config.model_id}")
print(f"Train file: {config.train_file}")
print(f"Val file: {config.val_file}")
print(f"Target modules: {config.target_modules}")

In [ ]:
# Step 5: Execute QLoRA Fine-Tuning & Capture Loss History
import time
from chess_commentator.training.trainer import run_qlora_training

start_time = time.time()
metrics = run_qlora_training(config)
training_duration = time.time() - start_time

print("Training Complete!")
print(f"Duration: {training_duration:.2f} seconds ({training_duration / 60:.2f} minutes)")
print("Metrics:", metrics)

In [ ]:
# Step 6: Post-Training Smoke Test Across Diverse Categories
import json, os, shutil
import pandas as pd
import torch
from chess_commentator.serving.engine import CommentaryInferenceEngine
from chess_commentator.dataset.checker import validate_chess_grounding
from chess_commentator.dataset.filters import validate_no_scaffolding_leakage

print("Loading fine-tuned model for smoke test...")
engine = CommentaryInferenceEngine(
    base_model_id=config.model_id,
    adapter_path=config.output_dir,
    load_in_4bit=True,
)
engine.load()

with open("data/smoke_test_v8_inputs.json") as f:
    smoke_positions = json.load(f)

results = []
for p in smoke_positions:
    prompt = p["full_serving_prompt"]
    comm = engine.generate(prompt)
    chk_grounding = validate_chess_grounding(
        fen=p["fen"],
        move_uci=p["move_uci"],
        commentary=comm,
        best_move_uci=p.get("best_move"),
    )
    chk_leakage = validate_no_scaffolding_leakage(comm)
    p_res = dict(p)
    p_res["generated_commentary"] = comm
    p_res["grounding_passed"] = chk_grounding.passed
    p_res["grounding_reason"] = chk_grounding.reason
    p_res["leakage_passed"] = chk_leakage.passed
    p_res["leakage_reason"] = chk_leakage.reason
    results.append(p_res)
    print("-" * 50)
    print(f"[{p['category']}] {p['move_san']} ({p['move_uci']})")
    print(f"Commentary: {comm}")
    print(f"Scaffolding Leak: {'NO' if chk_leakage.passed else 'YES: ' + chk_leakage.reason}")
    print(f"Grounding Passed: {'YES' if chk_grounding.passed else 'NO: ' + chk_grounding.reason}")

out_json_path = "/kaggle/working/smoke_test_results.json"
with open(out_json_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved {out_json_path}")

clean_adapter_dir = "/kaggle/working/phi35_adapter_clean"
os.makedirs(clean_adapter_dir, exist_ok=True)
for item in os.listdir(config.output_dir):
    s = os.path.join(config.output_dir, item)
    d = os.path.join(clean_adapter_dir, item)
    if os.path.isfile(s):
        shutil.copy2(s, d)
with open(os.path.join(clean_adapter_dir, "dataset-metadata.json"), "w") as f:
    json.dump({"title": "phi35-chess-adapter-v10", "id": "miteshsingh7/phi35-chess-adapter-v10", "licenses": [{"name": "MIT"}]}, f, indent=2)
print(f"Prepared clean adapter directory at {clean_adapter_dir}")

df_summary = pd.DataFrame([{
    "Category": r["category"],
    "Move": r["move_san"],
    "Leakage Clean": "YES" if r["leakage_passed"] else "LEAK",
    "Grounding": "PASS" if r["grounding_passed"] else "FAIL",
    "Reason": r["grounding_reason"],
} for r in results])
print("\n" + "=" * 60)
print("SMOKE TEST SUMMARY EVALUATION TABLE")
print("=" * 60)
print(df_summary.to_string(index=False))
total_clean = sum(r["leakage_passed"] for r in results)
total_grounding = sum(r["grounding_passed"] for r in results)
print(f"\nScaffolding Clean Rate: {total_clean} / {len(results)}")
print(f"Grounding Pass Rate:    {total_grounding} / {len(results)}")
